In [ ]:
import os
import subprocess

# If running in Colab, clone the repo and enter it to access reference_implementation
if 'COLAB_RELEASE_TAG' in os.environ:
    if not os.path.isdir('hetnet-traffic-forecast'):
        print('Cloning repository...')
        subprocess.check_call(['git', 'clone', 'https://github.com/Sreekar-2612/hetnet-traffic-forecast.git'])
    
    # Move into the repo directory so the notebook can find reference_implementation/
    os.chdir('/content/hetnet-traffic-forecast')
    print('Colab Environment Ready! You can now run the rest of the cells.')


# TASTF: Tier-Aware Spatiotemporal Forecasting for HetNets

This notebook implements the **TASTF** (Tier-Aware Spatiotemporal Forecasting) model for Heterogeneous Cellular Networks (HetNets). 

### Core Contribution
TASTF uses a **Heterogeneous Graph Convolutional Network** that treats Macro, Pico, and Femto base stations as distinct node types with type-specific convolutional layers via PyTorch Geometric `HeteroConv`. This approach explicitly models the hierarchy and cross-tier influence in HetNets, which is then processed by a **Transformer Encoder** to capture long-range temporal dependencies.

---

## 1. Environment Setup & Dependencies

In [1]:
# Pre-built wheels (data.pyg.org) — avoids 20+ min source builds for torch-scatter/torch-sparse.
import subprocess
import sys
import torch

def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip_install("pandas", "numpy", "scikit-learn", "matplotlib")

t = torch.__version__.split("+")[0]
cuda = ("cu" + torch.version.cuda.replace(".", "")) if torch.version.cuda else "cpu"
index_url = f"https://data.pyg.org/whl/torch-{t}+{cuda}.html"
print(f"Installing PyG extensions from: torch-{t}+{cuda}")

try:
    pip_install("torch-scatter", "torch-sparse", "-f", index_url)
except subprocess.CalledProcessError:
    # If your exact torch patch isn't published, try X.Y.0 wheels (same major.minor).
    parts = t.split(".")
    if len(parts) >= 3:
        t_alt = f"{parts[0]}.{parts[1]}.0"
        index_url = f"https://data.pyg.org/whl/torch-{t_alt}+{cuda}.html"
        print(f"Retrying with: torch-{t_alt}+{cuda}")
        pip_install("torch-scatter", "torch-sparse", "-f", index_url)
    else:
        raise

pip_install("torch-geometric")
print("Done.")

Installing PyG extensions from: torch-2.10.0+cu128
Done.


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from torch_geometric.nn import HeteroConv, SAGEConv
from torch_geometric.data import HeteroData

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

Using device: cuda


## 2. Training (canonical pipeline)

Runs the same code as 
eference_implementation/train.py: baselines, chronological split, temporal features, inverse metrics.  
**Data:** place Milan *.txt files in hetnet-traffic-forecast/Wireless Dataset/ (or set DATA_DIR below).


In [3]:
import os
import sys
from pathlib import Path

# Debugging information
print(f"Current working directory: {Path.cwd()}")
print(f"Notebook file resolve: {Path('TASTF_Implementation.ipynb').resolve()}")

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / 'hetnet-traffic-forecast',
    Path(r'c:/Users/sreek/OneDrive/Desktop/ACADS-ASSIGNS/wireless/project'),
    Path('/mnt/c/Users/sreek/OneDrive/Desktop/ACADS-ASSIGNS/wireless/project'),
    # Try to find it relative to the notebook if it's in a subfolder or something
    Path(__file__).parent if '__file__' in locals() else Path('.'),
]

found = False
for root in candidates:
    ref = root / 'reference_implementation'
    print(f"Checking: {ref}")
    if ref.is_dir():
        print(f"Success! Found at: {ref}")
        if str(ref) not in sys.path:
            sys.path.insert(0, str(ref))
        REPO = root
        found = True
        break

if not found:
    # Aggressive search in parent directories
    curr = Path.cwd().resolve()
    for _ in range(5):
        ref = curr / 'reference_implementation'
        print(f"Checking parent: {ref}")
        if ref.is_dir():
            print(f"Success! Found at: {ref}")
            if str(ref) not in sys.path:
                sys.path.insert(0, str(ref))
            REPO = curr
            found = True
            break
        if curr == curr.parent:
            break
        curr = curr.parent

if not found:
    raise FileNotFoundError('reference_implementation/ not found. Diagnostic: ' + 
                            f'CWD={Path.cwd()} | candidates={[str(c) for c in candidates]}')

from train import run_training
from paths import resolve_wireless_dataset_dir

DATA_DIR = os.environ.get('TASTF_DATA_DIR') or resolve_wireless_dataset_dir(None)
print('DATA_DIR:', DATA_DIR)

run_training(
    data_dir=DATA_DIR,
    epochs=50,
    batch_size=32,
    seed=42,
)


Current working directory: /content
Notebook file resolve: /content/TASTF_Implementation.ipynb
Checking: /content/reference_implementation
Checking: /reference_implementation
Checking: /content/hetnet-traffic-forecast/reference_implementation
Checking: c:/Users/sreek/OneDrive/Desktop/ACADS-ASSIGNS/wireless/project/reference_implementation
Checking: /mnt/c/Users/sreek/OneDrive/Desktop/ACADS-ASSIGNS/wireless/project/reference_implementation
Checking: reference_implementation
Checking parent: /content/reference_implementation
Checking parent: /reference_implementation


FileNotFoundError: reference_implementation/ not found. Diagnostic: CWD=/content | candidates=['/content', '/', '/content/hetnet-traffic-forecast', 'c:/Users/sreek/OneDrive/Desktop/ACADS-ASSIGNS/wireless/project', '/mnt/c/Users/sreek/OneDrive/Desktop/ACADS-ASSIGNS/wireless/project', '.']

## 3. Metrics & figures

Requires 
eference_implementation/results.npz from the training cell.


In [ ]:
import os
import sys
from pathlib import Path

# Debugging information
print(f"Current working directory: {Path.cwd()}")
print(f"Notebook file resolve: {Path('TASTF_Implementation.ipynb').resolve()}")

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / 'hetnet-traffic-forecast',
    Path(r'c:/Users/sreek/OneDrive/Desktop/ACADS-ASSIGNS/wireless/project'),
    Path('/mnt/c/Users/sreek/OneDrive/Desktop/ACADS-ASSIGNS/wireless/project'),
    # Try to find it relative to the notebook if it's in a subfolder or something
    Path(__file__).parent if '__file__' in locals() else Path('.'),
]

found = False
for root in candidates:
    ref = root / 'reference_implementation'
    print(f"Checking: {ref}")
    if ref.is_dir():
        print(f"Success! Found at: {ref}")
        if str(ref) not in sys.path:
            sys.path.insert(0, str(ref))
        REPO = root
        found = True
        break

if not found:
    # Aggressive search in parent directories
    curr = Path.cwd().resolve()
    for _ in range(5):
        ref = curr / 'reference_implementation'
        print(f"Checking parent: {ref}")
        if ref.is_dir():
            print(f"Success! Found at: {ref}")
            if str(ref) not in sys.path:
                sys.path.insert(0, str(ref))
            REPO = curr
            found = True
            break
        if curr == curr.parent:
            break
        curr = curr.parent

if not found:
    raise FileNotFoundError('reference_implementation/ not found. Diagnostic: ' + 
                            f'CWD={Path.cwd()} | candidates={[str(c) for c in candidates]}')

from train import run_training
from paths import resolve_wireless_dataset_dir

DATA_DIR = os.environ.get('TASTF_DATA_DIR') or resolve_wireless_dataset_dir(None)
print('DATA_DIR:', DATA_DIR)

run_training(
    data_dir=DATA_DIR,
    epochs=50,
    batch_size=32,
    seed=42,
)
